# Using Machine Learning techniques to predict oral cancer

## Imports

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

## Load in data, basic data information

In [ ]:
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")

file_path = os.path.join(path, "healthcare-dataset-stroke-data.csv")
df = pd.read_csv(file_path)

df.head()

In [ ]:
df.info()

## Data cleaning

In [ ]:
df.columns

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [ ]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

### Adjusting column types

In [ ]:
df["age"] = df["age"].astype("uint8")
df

### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [ ]:
df.isna().any().any()

### Dealing with NaNs

BMI is the only columns with missing values. We fill them using the gender average.

In [ ]:
df["bmi"] = df["bmi"].fillna(df.groupby("gender")["bmi"].transform("mean"))
df

### Modifying the data

In [ ]:
# Binary to 1/0
df["ever_married"] = df["ever_married"].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["gender", "work_type", "Residence_type", "smoking_status"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

In [ ]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "stroke"] + ["stroke"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [ ]:
def get_renamed_column(name):
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [ ]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

### Check for duplicates

In [ ]:
any(df.duplicated())

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

There should be a similar number of records in each class.

In [ ]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

## Preparing the train and test sets

### Train-test split

In [ ]:
columns_to_drop = []
df = df.drop(columns=columns_to_drop)

In [ ]:
X = df.drop(columns=[df.columns[-1]]) 
y = df[df.columns[-1]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Feature scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Create and train models

### Random Forests

In [ ]:
random_forest_models = {
    "RandomForest_100_5": RandomForestClassifier(
        n_estimators=100,
        max_depth=5
    ),
    "RandomForest_50_5": RandomForestClassifier(
        n_estimators=50,
        max_depth=5
    ),
    "RandomForest_50_10": RandomForestClassifier(
        n_estimators=50,
        max_depth=10
    ),
    "RandomForest_500_3": RandomForestClassifier(
        n_estimators=500,
        max_depth=3
    ),
    "RandomForest_200_10": RandomForestClassifier(
        n_estimators=200,
        max_depth=10
    ),
    "RandomForest_300_7": RandomForestClassifier(
        n_estimators=300,
        max_depth=7
    ),
}

In [ ]:
random_forest_scores = {}
for name, model in random_forest_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    scores = {"Accuracy": accuracy, "Precision": precision, "Recall": recall}
    random_forest_scores[name] = scores

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}\n")

print(random_forest_scores)